# st_pval — the p-value as a proportion: False-Positive Psychology

**Paper:** Simmons, Nelson, Simonsohn (2011). *False-Positive Psychology: Undisclosed
Flexibility in Data Collection and Analysis Allows Presenting Anything as Significant.*
Psychological Science 22(11), 1359–1366. One of the most cited methodology papers of the
decade and a trigger of the replication crisis.

**What we build:** our own permutation test and t-test, a Monte Carlo estimate of the
false-positive rate, the four "researcher degrees of freedom" from Table 1, and their
combination. There are no trainable models: the core of the paper is a simulation, and that
simulation is its protocol.

**Compute:** CPU, all simulations together take ~20 seconds.

**Parts:**
1. The p-value by hand — permutation test
2. The t-test — an analytic shortcut
3. A Monte Carlo FPR machine
4. Researcher degrees of freedom A–D (Table 1)
5. The combined p-hacker + the key figure

## Experiments first

Open the paper (it is short and blunt) and fill this in by hand BEFORE reading further into the method:

- **What was actually shown** vs what the abstract promises:

- **Weakest experiment / assumption:**

- **Where it breaks first** (limits of the method):


## Why this matters

The $p<0.05$ threshold is presented as protection against false discoveries: at most a 5%
chance of seeing an effect where there is none. Simmons and colleagues showed that the
guarantee holds only for an analysis fixed before the data are collected. Allow yourself the
usual working liberties — a second dependent variable, "I'll collect a few more participants",
a covariate, picking a pair out of three conditions — and the actual chance of a false
discovery grows to 61%.

The paper's method: build a world where there is no effect by construction, run thousands of
"studies" with these liberties through it, and count the share that come out significant. The
advantage over verbal criticism of methodology is a number any skeptic can reproduce in a
minute. Here you are the skeptic.

Data: the null world is generated, as in the paper's protocol. The permutation test is also
checked on Student's 1908 data (sleep gain under two hypnotics), the real experiment the
t-test itself started from.

## Theory (Feynman-style)

### The p-value by hand

We have two groups of numbers and observe a difference in means $\Delta_{obs}$. The question
the p-value answers: **if there were actually no difference, how often would chance produce
$|\Delta| \ge |\Delta_{obs}|$?**

The permutation answer is literal. If the groups are indistinguishable, the labels "group 1 /
group 2" are decoration: pool the samples, shuffle the labels, recompute $\Delta$, repeat $N$
times. The share of permutations with $|\Delta| \ge |\Delta_{obs}|$ is the p-value. That is the
whole definition, with no formula about distributions.

The t-test is an analytic shortcut for the same idea: the shape of the null distribution is
known in advance,

$$t = \frac{\bar x - \bar y}{s_p\sqrt{1/n_x + 1/n_y}},\qquad
s_p^2 = \frac{(n_x-1)s_x^2 + (n_y-1)s_y^2}{n_x+n_y-2},$$

and the tail is computed from Student's distribution with $n_x{+}n_y{-}2$ degrees of freedom,
so nothing needs to be shuffled.

### Calibration and how it breaks

Under $H_0$ the p-value is uniform on $[0,1]$, so the rule "reject when $p<\alpha$" is wrong in
exactly a fraction $\alpha$ of null worlds. This is a property of the procedure, and it is
fragile: compute several p-values and report the smallest, and the minimum of uniform
variables is no longer uniform:

$$P\big(\min(p_1,\dots,p_k)<\alpha\big) = 1-(1-\alpha)^k \quad \text{under independence}.$$

When the tests are correlated the inflation is smaller, but still above $\alpha$, and a formula
no longer gives it. The whole paper measures this inflation by simulation in four realistic
scenarios.

**Where my gap is:** be honest about which of these you could have explained to a colleague yesterday without looking.

❓ **Inline Question 0.** A researcher runs exactly ONE t-test per study, no min-p, but
publishes only the significant results and files the rest away. The FPR of each individual
study is still 5%. What happens to the share of false results among the PUBLISHED ones? How
does this mechanism differ from min-p, and how is it similar?

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Real data: Student (1908), The probable error of a mean.
# Sleep gain (hours) under two isomers of hyoscyamine. The same 10 patients took both,
# so the data are paired; Parts 1-2 treat them as two independent groups on purpose (see Inline Question 1b).
dextro = np.array([0.7, -1.6, -0.2, -1.2, -0.1, 3.4, 3.7, 0.8, 0.0, 2.0])
laevo  = np.array([1.9,  0.8,  1.1,  0.1, -0.1, 4.4, 5.5, 1.6, 4.6, 3.4])

ALPHA = 0.05
print("group means:", dextro.mean(), laevo.mean())

## Part 1. The p-value by hand — permutation test

Implement a two-sided permutation test for the difference in means. On Student's data you
should get $p \approx 0.08$: "almost significant", a convenient point for asking awkward questions.

Technical convention: Laplace smoothing $\frac{\#\{\text{as or more extreme}\}+1}{N+1}$, so the
estimate is never zero for finite $N$.

In [ ]:
def perm_pvalue(x, y, n_perm=10000, rng=None):
    """Two-sided permutation test for the difference in means of two groups.

    Args:
        x, y: 1d float arrays — two independent samples.
        n_perm: int — number of random permutations.
        rng: np.random.Generator.
    Returns:
        float in (0, 1] — share of permutations of the pooled sample that give
        |difference in means| at least as large as the observed one; with +1/+1
        smoothing, so it is never exactly 0.
    """
    rng = rng or np.random.default_rng(0)
    # YOUR CODE
    raise NotImplementedError

In [ ]:
pp = perm_pvalue(dextro, laevo, 20000, np.random.default_rng(0))
pt_ref = stats.ttest_ind(dextro, laevo).pvalue
assert 0 < pp <= 1, "the p-value must lie in (0,1] — check the smoothing"
assert abs(pp - pt_ref) < 0.02, (
    f"perm={pp:.4f} vs t-test={pt_ref:.4f}: on this data both estimates of the same tail "
    "should agree; if they diverge, the absolute value (two-sidedness) is most often missing")
p_shift = perm_pvalue(dextro + 10.0, laevo, 20000, np.random.default_rng(0))
assert p_shift < 0.001, "a +10 hour shift in sleep must be overwhelmingly significant — check the comparison sign"
print(f"ok: perm p = {pp:.4f}  (Expected ≈ 0.080)")

❓ **Inline Question 1.** Make the test one-sided ("the laevo isomer is better") and p halves
to ~0.04, which counts as a "discovery". Who should fix the direction of the hypothesis, and
at what point, for this not to be cheating? How does this relate to the main topic of the paper?

❓ **Inline Question 1b.** The same ten patients took both isomers, so these are paired measurements. A paired t-test (`stats.ttest_rel`) on the same numbers gives $p \approx 0.003$, against $\approx 0.08$ for the independent-groups test we just matched. Which test is correct here, and why? Now imagine a researcher who runs both and reports whichever looks better: which of the paper's researcher degrees of freedom is that, and what would have to be written down before the data were seen to rule it out?

## Part 2. The t-test — an analytic shortcut

The same question about the tail, only now the null distribution is known in closed form.
Implement the two-sample pooled-variance t-test (the formula is in the theory above) and match
scipy to machine precision.

In [ ]:
def t_pvalue(x, y):
    """Two-sample t-test (pooled variance), two-sided p.

    Args:
        x, y: 1d float arrays.
    Returns:
        float — p-value; matches stats.ttest_ind(x, y).pvalue to ~1e-12.
    Note: take the tail from stats.t (this function is the only place scipy is needed).
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
pt = t_pvalue(dextro, laevo)
assert abs(pt - stats.ttest_ind(dextro, laevo).pvalue) < 1e-10, (
    "does not match scipy: check the pooled variance (ddof=1) and the degrees of freedom n_x+n_y-2")
assert abs(t_pvalue(dextro, laevo) - t_pvalue(laevo, dextro)) < 1e-12, "the test must be symmetric"
rng_t = np.random.default_rng(7)
big = t_pvalue(rng_t.standard_normal(500) + 1.0, rng_t.standard_normal(500))
assert big < 1e-10, "a 1σ effect at n=500 must be overwhelmingly significant"
print(f"ok: t p = {pt:.4f}  (Expected ≈ 0.0792)")

❓ **Inline Question 2.** The pooled variance assumes equal group variances. Give an example
of data where this assumption is false, and say how it is fixed (hint: Welch). Why is the
difference between the two variants barely visible when the groups have equal sizes?

## Part 3. A Monte Carlo FPR machine

The core of the paper is a counter: how many "null" studies declared a discovery.
`experiment_fn(rng)` returns the p-value of one simulated study (if it runs several analyses
inside, the p the author would report). This machine will then run every scenario of Table 1.

Honest baseline: two conditions of $n=20$ each from $\mathcal N(0,1)$, one t-test. Expectation:
FPR equals $\alpha$, since the test is calibrated by construction.

In [ ]:
def fpr_mc(experiment_fn, n_sims, rng, alpha=ALPHA):
    """Monte Carlo estimate of a procedure's false-positive rate.

    Args:
        experiment_fn: callable(rng) -> float — p-value of one null study.
        n_sims: int — number of simulated studies.
        rng: np.random.Generator — one for the whole run.
        alpha: float — significance threshold.
    Returns:
        float — share of studies with p < alpha.
    """
    # YOUR CODE
    raise NotImplementedError


def honest(rng, n=20):
    """One honest null study: two N(0,1) conditions of n each, one t-test.

    Returns:
        float — p-value of that single test.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
r_honest = fpr_mc(honest, 10000, np.random.default_rng(1))
assert 0.035 < r_honest < 0.065, (
    f"FPR of the honest test = {r_honest:.3f}, expected ≈0.05: noticeably higher — min-p is leaking "
    "somewhere or rng is being reused; lower — p is computed conservatively")
print(f"ok: honest FPR = {r_honest:.3f}  (Expected ≈ 0.050)")

❓ **Inline Question 3.** Is FPR≈α a property of the data or of the procedure? What happens
to the FPR of the honest test at $n=200$ instead of $20$? And what happens to power (the chance
of catching a real effect)? The two answers differ — why?

## Part 4. Researcher degrees of freedom (Table 1)

Four ordinary liberties, each innocent-looking. The protocols are from the paper:

- **A. Two dependent variables** (correlation $r=0.5$): analyze DV1, DV2 and their mean;
  the best result goes into the paper.
- **B. Collecting more observations:** look at $n=20$ per group; if not significant, add 10
  more and look again. The result is "significant" if either look gave $p<.05$.
- **C. A covariate (gender):** t-test, ANCOVA with gender, a model with a gender×condition
  interaction; the best of the condition/interaction effects is reported. — *provided below,
  already written: the OLS plumbing is not part of the core, and by then you will have written
  the min-p pattern yourself twice.*
- **D. Three conditions:** collect three groups, report any significant pair.

Simulate A, B, D. Each function is ONE null study and returns the p the author would report.

In [ ]:
def dof_two_dvs(rng, n=20, r=0.5):
    """A: one null study with two correlated DVs and a flexible choice of result.

    Args:
        rng: np.random.Generator; n: int — size of each of the two groups;
        r: float — within-subject correlation of the DVs.
    Returns:
        float — the p the author would report (the worst case for honesty).
    Note: both groups are drawn from the same bivariate null distribution.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
r_a = fpr_mc(dof_two_dvs, 10000, np.random.default_rng(2))
assert 0.085 < r_a < 0.125, (
    f"FPR(A) = {r_a:.3f}, expected ≈0.10 (paper: 9.5%): much lower — one of the three "
    "analyses got lost; ~0.14 — the DVs were generated independently, the r=0.5 correlation never made it into the data")
print(f"ok: A two-DVs FPR = {r_a:.3f}  (Expected ≈ 0.10, paper 0.095)")

In [ ]:
def dof_optional_stopping(rng, n1=20, add=10):
    """B: one null study with optional stopping — look at n1, on failure add `add` and look again.

    Args:
        rng; n1: int — initial group size; add: int — observations added per group.
    Returns:
        float — the p the author would report.
    Note: the added observations are NEW draws; the initial ones stay in the sample.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
r_b = fpr_mc(dof_optional_stopping, 10000, np.random.default_rng(3))
assert 0.060 < r_b < 0.100, (
    f"FPR(B) = {r_b:.3f}, expected ≈0.08 (paper: 7.7%): ~0.05 — the second look has no effect "
    "(did you regenerate the whole sample?); ~0.10 — the two looks are treated as independent")
print(f"ok: B optional stopping FPR = {r_b:.3f}  (Expected ≈ 0.080, paper 0.077)")

In [ ]:
def dof_three_conditions(rng, n=20):
    """D: one null study with three conditions and a choice of pair.

    Args:
        rng; n: int — size of each of the three groups.
    Returns:
        float — the p the author would report.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
r_d = fpr_mc(dof_three_conditions, 10000, np.random.default_rng(5))
assert 0.105 < r_d < 0.150, (
    f"FPR(D) = {r_d:.3f}, expected ≈0.126 (paper: 12.6%): check that there are exactly three pairs "
    "and that the groups do not reuse the same samples")
print(f"ok: D three conditions FPR = {r_d:.3f}  (Expected ≈ 0.126, paper 0.126)")

Scenario C is provided (plumbing). Read `dof_covariate` and find in it the same min-p
pattern you have already written twice yourself.

In [ ]:
def ols_pvalues(X, y):
    # p-values of OLS coefficients (two-sided, t-distribution); provided
    XtX_inv = np.linalg.inv(X.T @ X)
    beta = XtX_inv @ X.T @ y
    resid = y - X @ beta
    df = len(y) - X.shape[1]
    sigma2 = resid @ resid / df
    se = np.sqrt(np.diag(XtX_inv) * sigma2)
    return 2 * stats.t.sf(np.abs(beta / se), df)


def dof_covariate(rng, n=20):
    # C: gender as a covariate — t-test, ANCOVA, interaction; min p over condition/interaction
    y = rng.standard_normal(2 * n)
    cond = np.r_[np.zeros(n), np.ones(n)]
    gender = np.tile(np.r_[np.zeros(n // 2), np.ones(n // 2)], 2)
    ps = [t_pvalue(y[cond == 0], y[cond == 1])]
    X1 = np.c_[np.ones(2 * n), cond, gender]
    ps.append(ols_pvalues(X1, y)[1])
    X2 = np.c_[np.ones(2 * n), cond, gender, cond * gender]
    p2 = ols_pvalues(X2, y)
    ps += [p2[1], p2[3]]
    return min(ps)


r_c = fpr_mc(dof_covariate, 10000, np.random.default_rng(4))
assert 0.090 < r_c < 0.135, f"FPR(C) = {r_c:.3f}, expected ≈0.113"
print(f"ok: C covariate FPR = {r_c:.3f}  (Expected ≈ 0.113, paper 0.117)")

❓ **Inline Question 4.** For three independent tests $1-(1-.05)^3 = 14.3\%$, yet scenario A
came out at ~10%. Why does correlation between the analyses REDUCE the inflation? And why does
"collecting more data" (scenario B) break calibration even though there is only more data?

## Part 5. The combined p-hacker and the key figure

Now everything at once, as in the last row of Table 1: three conditions × three DV variants
(DV1, DV2, mean) × four analyses (t-test, ANCOVA, condition in the interaction model, the
interaction itself) × two-stage optional stopping. 36 analyses per stage; the second stage runs
only if the first found nothing. The paper's claim: **60.7%**.

In [ ]:
def hacker_combined(rng, n1=20, add=10, r=0.5):
    """All four liberties at once — one null "study" by a careless author.

    Args:
        rng; n1: int — initial size of each of the three groups; add: int — observations added;
        r: float — correlation of the two DVs.
    Returns:
        float — the p the author would report.
    Note: gender is balanced within each group; added observations are new draws appended to the old ones.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
r_comb = fpr_mc(hacker_combined, 4000, np.random.default_rng(6))
assert 0.55 < r_comb < 0.66, (
    f"FPR(combined) = {r_comb:.3f}, the paper claims 60.7%: much lower — one of the liberties "
    "is not taking part (most often the DV variants or the second stage are forgotten); higher — min is taken "
    "over extra OLS coefficients (intercept, gender)")
assert r_comb > 8 * r_honest, "the combined effect must be qualitatively different from the honest 5%"
print(f"ok: combined FPR = {r_comb:.3f}  (Expected ≈ 0.60, paper 0.607)")

### The key figure — your own Table 1

In [ ]:
labels = ["honest", "A: 2 DVs", "B: more data", "C: covariate", "D: 3 conditions", "A+B+C+D"]
mine  = [r_honest, r_a, r_b, r_c, r_d, r_comb]
paper = [0.05, 0.095, 0.077, 0.117, 0.126, 0.607]

fig, ax = plt.subplots(figsize=(8, 4))
xs = np.arange(len(labels))
ax.bar(xs, mine, width=0.6, label="my simulation")
ax.scatter(xs, paper, zorder=3, marker="D", label="Simmons et al., Table 1")
ax.axhline(ALPHA, ls="--", lw=1)
ax.set_xticks(xs, labels, rotation=20)
ax.set_ylabel("false-positive rate")
ax.legend(); ax.set_title("FPR inflation from researcher degrees of freedom")
plt.tight_layout(); plt.show()

for l, m, p in zip(labels, mine, paper):
    print(f"{l:16s} mine={m:5.3f}  paper={p:5.3f}  Δ={m-p:+.3f}")

Each bar should cover its diamond up to Monte Carlo noise. The dashed line at 0.05 is the
promise that "$p<.05$" makes. The combined bar shows what that promise is worth when the
analysis is not fixed in advance: a coin flip would be more honest, since a coin at least
states its 50% openly.

❓ **Inline Question 5 (recognition).** You compared 12 model variants on one test set, and the
best one "significantly" beat the baseline. How many p-values did you silently minimize? In
what other situation from your own practice is it worth running a null-world simulation like
this before believing your discovery?

### A note on the fragile spot

The most common mistake in B and in the combined scenario is regenerating the whole sample when
adding data. That is a different protocol: two independent studies instead of peeking, and it
has a different FPR. Optional stopping means the old observations stay and new ones are drawn
from the same rng.

## Reflection

- **What I take into my work** (one concrete technique):

- **In what NEW situation I will reach for a null-world simulation:**

- **What is still unclear** → what to read next: `st_power` (how much data you need to catch a
  real effect), `st_mult` (Bonferroni/FDR, the legitimate fix for min-p), `st_ci`
  (bootstrap, an interval instead of a verdict):


## Bonus: Going deeper (ungraded)

1. Add a Bonferroni correction over all 72 analyses to `hacker_combined`. Did the FPR return to
   5%? Estimate what would happen to power under a real effect, and why Bonferroni is nearly
   free here but a disaster in genomics (a million tests).
2. Publication filter: the authors are honest (one test), but the journal prints only $p<.05$.
   With 10% of effects being real and power 0.5, compute the share of false results among the
   published ones.
3. Legitimate optional stopping: alpha spending (Pocock). Implement two looks with a threshold
   of 0.0294 at each and show by simulation that the overall FPR ≈ 5%.

**Experiment log:**


---
## SPOILERS

Full solutions. Start on your own; come here when you are seriously stuck, or to compare afterwards.

<details><summary>Part 1: perm_pvalue</summary>

```python
def perm_pvalue(x, y, n_perm=10000, rng=None):
    rng = rng or np.random.default_rng(0)
    obs = abs(x.mean() - y.mean())
    z = np.concatenate([x, y])
    cnt = 0
    for _ in range(n_perm):
        p = rng.permutation(z)
        cnt += abs(p[:len(x)].mean() - p[len(x):].mean()) >= obs
    return (cnt + 1) / (n_perm + 1)
```
</details>

<details><summary>Part 2: t_pvalue</summary>

```python
def t_pvalue(x, y):
    nx, ny = len(x), len(y)
    sp2 = ((nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1)) / (nx + ny - 2)
    t = (x.mean() - y.mean()) / np.sqrt(sp2 * (1 / nx + 1 / ny))
    return 2 * stats.t.sf(abs(t), nx + ny - 2)
```
</details>

<details><summary>Part 3: fpr_mc + honest</summary>

```python
def fpr_mc(experiment_fn, n_sims, rng, alpha=ALPHA):
    hits = 0
    for _ in range(n_sims):
        hits += experiment_fn(rng) < alpha
    return hits / n_sims

def honest(rng, n=20):
    return t_pvalue(rng.standard_normal(n), rng.standard_normal(n))
```
</details>

<details><summary>Part 4: A, B, D</summary>

```python
def dof_two_dvs(rng, n=20, r=0.5):
    cov = [[1, r], [r, 1]]
    a = rng.multivariate_normal([0, 0], cov, n)
    b = rng.multivariate_normal([0, 0], cov, n)
    return min(t_pvalue(a[:, 0], b[:, 0]), t_pvalue(a[:, 1], b[:, 1]),
               t_pvalue(a.mean(1), b.mean(1)))

def dof_optional_stopping(rng, n1=20, add=10):
    x, y = rng.standard_normal(n1), rng.standard_normal(n1)
    p1 = t_pvalue(x, y)
    if p1 < 0.05:
        return p1
    x2 = np.concatenate([x, rng.standard_normal(add)])
    y2 = np.concatenate([y, rng.standard_normal(add)])
    return min(p1, t_pvalue(x2, y2))

def dof_three_conditions(rng, n=20):
    g = [rng.standard_normal(n) for _ in range(3)]
    return min(t_pvalue(g[i], g[j]) for i in range(3) for j in range(i + 1, 3))
```
</details>

<details><summary>Part 5: hacker_combined</summary>

```python
def hacker_combined(rng, n1=20, add=10, r=0.5):
    cov = [[1, r], [r, 1]]

    def stage_minp(groups, genders):
        ps = []
        for i in range(3):
            for j in range(i + 1, 3):
                gi, gj = groups[i], groups[j]
                for dv_i, dv_j in [(gi[:, 0], gj[:, 0]), (gi[:, 1], gj[:, 1]),
                                   (gi.mean(1), gj.mean(1))]:
                    y = np.concatenate([dv_i, dv_j])
                    cnd = np.r_[np.zeros(len(dv_i)), np.ones(len(dv_j))]
                    gen = np.concatenate([genders[i], genders[j]])
                    ps.append(t_pvalue(dv_i, dv_j))
                    X1 = np.c_[np.ones(len(y)), cnd, gen]
                    ps.append(ols_pvalues(X1, y)[1])
                    X2 = np.c_[np.ones(len(y)), cnd, gen, cnd * gen]
                    p2 = ols_pvalues(X2, y)
                    ps += [p2[1], p2[3]]
        return min(ps)

    groups = [rng.multivariate_normal([0, 0], cov, n1) for _ in range(3)]
    genders = [np.r_[np.zeros(n1 // 2), np.ones(n1 // 2)] for _ in range(3)]
    p1 = stage_minp(groups, genders)
    if p1 < 0.05:
        return p1
    groups = [np.vstack([g, rng.multivariate_normal([0, 0], cov, add)]) for g in groups]
    genders = [np.r_[g, np.zeros(add // 2), np.ones(add - add // 2)] for g in genders]
    return min(p1, stage_minp(groups, genders))
```
</details>
